RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [38]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [39]:
# Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        
        try:
            loader = PyPDFLoader(pdf_file)
            documents = loader.load()
            
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            
            all_documents.extend(documents)
            print(f"✅ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"❌ Error: {e}")
            
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: decision_tree.pdf
✅ Loaded 9 pages

Processing: ensemble_learning.pdf
✅ Loaded 7 pages

Processing: logistic_regression.pdf
✅ Loaded 7 pages

Processing: random_forest.pdf
✅ Loaded 6 pages

Total documents loaded: 29


In [40]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m85', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36', 'creationdate': '2020-09-23T09:27:39+00:00', 'moddate': '2020-09-23T09:27:39+00:00', 'source': '..\\data\\pdf\\decision_tree.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'decision_tree.pdf', 'file_type': 'pdf'}, page_content='Decision Tree\nDecision tree is one of the most popular machine learning algorithms used all along.\nDecision trees are used for both classification and regression problems.\n1. Why Decision trees?\nWe have couple of other algorithms there, so why do we have to choose Decision trees??\nDecision tress often mimic the human level thinking so its so simple to understand the data and make\nsome good interpretations.\nDecision trees actually make you see the logic for the data to interpret(not like black box algorithms\nlike SVM,NN,etc..)\nA decision tree follows a set o

In [41]:
# Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    
    """Split documents into smaller chunks for better RAG Performance"""
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        # seperators=["\n\n", "\n", "", " "]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    
    return split_docs
    

In [42]:
chunks=split_documents(all_pdf_documents)

Split 29 documents into 51 chunks

Example chunk:
Content: Decision Tree
Decision tree is one of the most popular machine learning algorithms used all along.
Decision trees are used for both classification and regression problems.
1. Why Decision trees?
We ha...
Metadata: {'producer': 'Skia/PDF m85', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36', 'creationdate': '2020-09-23T09:27:39+00:00', 'moddate': '2020-09-23T09:27:39+00:00', 'source': '..\\data\\pdf\\decision_tree.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'decision_tree.pdf', 'file_type': 'pdf'}


In [43]:
chunks

[Document(metadata={'producer': 'Skia/PDF m85', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36', 'creationdate': '2020-09-23T09:27:39+00:00', 'moddate': '2020-09-23T09:27:39+00:00', 'source': '..\\data\\pdf\\decision_tree.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'decision_tree.pdf', 'file_type': 'pdf'}, page_content='Decision Tree\nDecision tree is one of the most popular machine learning algorithms used all along.\nDecision trees are used for both classification and regression problems.\n1. Why Decision trees?\nWe have couple of other algorithms there, so why do we have to choose Decision trees??\nDecision tress often mimic the human level thinking so its so simple to understand the data and make\nsome good interpretations.\nDecision trees actually make you see the logic for the data to interpret(not like black box algorithms\nlike SVM,NN,etc..)\nA decision tree follows a set o

Embedding and VectorStore DB

In [44]:
import uuid
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.embeddings import ZhipuAIEmbeddings

In [45]:
class EmbeddingManager:
    
    "Handles document embedding generation using SentenceTransformer"
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        
        """
            Initialize the embedding manager
            
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise  
    
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generate embeddings for a list of texts
            
        Args:
            texts: List of text strings to embed
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

# initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager
        
    

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5262.87it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\User\AppData\Local\Temp\ipykernel_17812\3257974734.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [46]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 51


In [47]:
chunks

[Document(metadata={'producer': 'Skia/PDF m85', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36', 'creationdate': '2020-09-23T09:27:39+00:00', 'moddate': '2020-09-23T09:27:39+00:00', 'source': '..\\data\\pdf\\decision_tree.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'decision_tree.pdf', 'file_type': 'pdf'}, page_content='Decision Tree\nDecision tree is one of the most popular machine learning algorithms used all along.\nDecision trees are used for both classification and regression problems.\n1. Why Decision trees?\nWe have couple of other algorithms there, so why do we have to choose Decision trees??\nDecision tress often mimic the human level thinking so its so simple to understand the data and make\nsome good interpretations.\nDecision trees actually make you see the logic for the data to interpret(not like black box algorithms\nlike SVM,NN,etc..)\nA decision tree follows a set o

For the above 2 class understand it like this:
1. We created the Embedding Manager class which generates the Embeddings
2. We created the VectoreStore class which stores those embeddings in the vector store


Before this we wrote the function to split the PDFs (documents) into small chunks -> so that we can pass them inside Embedding manager to create embeddings -> which we can store inside Vectore Store.

In [48]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]


# Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)


# Store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 51 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.08s/it]

Generated embeddings with shape: (51, 384)
Adding 51 documents to vector store...
Successfully added 51 documents to vector store
Total documents in collection: 102


Retriever Pipeline From VectorStore